# import

In [5]:
import os
import random
import shutil
from pathlib import Path

import torch
from datasets import load_dataset, Dataset, concatenate_datasets

from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

# 토크나이저 fork 경고(데드락 방지)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Variable definition

In [ ]:
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
OUT_DIR  = "/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2" # 결과 저장 폴더

# Calibration dataset mix
MANTA_ID = "LGAI-EXAONE/MANTA-1M"
MANTA_SPLIT = "train"

GSM8K_ID = "gsm8k"
GSM8K_CONFIG = "main"
GSM8K_SPLIT = "train"

# Calibration
NUM_CALIBRATION_SAMPLES = 1024    # 512
MAX_SEQUENCE_LENGTH = 1024        # 512보다 일반적으로 안정적(단, 속도/메모리 비용 증가)

# Quantization schemes
SCHEME_MAIN = "W4A16"  # 대부분 레이어: GPTQ int4
SCHEME_HI   = "W8A16"  # 민감/에러 집중 레이어: int8

# vLLM 제출 호환을 위해 보통 lm_head는 fp16/bf16 유지 권장
# embed_tokens도 보통 보호 추천 (단, 규칙상 금지는 아님)
PROTECT_ALWAYS = ["lm_head", "embed_tokens"]

# 레이어 보호: "마지막 2개 레이어"
# EXAONE-4.0-1.2B가 30 layers라는 전제(일반적으로 0~29)
PROTECT_LAST_N_LAYERS = 2

# Dataset Load

In [7]:
# =========================================
# [블록 2] Dataset load + text normalize
# =========================================
def pick_text(example: dict) -> str:
    """
    서로 다른 데이터셋에서 텍스트 컬럼이 다를 수 있으니 최대한 안전하게 뽑는다.
    - MANTA: 보통 instruction/input/output 또는 messages 형태일 수 있음
    - GSM8K: question, answer
    """
    # 1) 가장 흔한 단일 텍스트 컬럼들
    for k in ["text", "content", "prompt"]:
        if k in example and isinstance(example[k], str) and example[k].strip():
            return example[k].strip()

    # 2) instruction style
    if "instruction" in example:
        ins = str(example.get("instruction", "")).strip()
        inp = str(example.get("input", "")).strip()
        out = str(example.get("output", "")).strip()
        s = ins
        if inp:
            s += "\n\n" + inp
        if out:
            s += "\n\n" + out
        return s.strip()

    # 3) chat/messages style
    if "messages" in example and isinstance(example["messages"], (list, tuple)):
        # 가능한 한 role/content를 직렬화
        parts = []
        for m in example["messages"]:
            if not isinstance(m, dict):
                continue
            role = str(m.get("role", "")).strip()
            content = str(m.get("content", "")).strip()
            if content:
                parts.append(f"[{role}] {content}" if role else content)
        if parts:
            return "\n".join(parts).strip()

    # 4) GSM8K
    if "question" in example:
        q = str(example.get("question", "")).strip()
        a = str(example.get("answer", "")).strip()
        if q and a:
            return f"Q: {q}\nA: {a}"
        if q:
            return f"Q: {q}"

    # 마지막 fallback: 문자열화
    return str(example)

# Calibration

In [8]:
# =========================================
# [블록 3] Build calibration dataset (80/20 mix)
# =========================================
SEED = 42
random.seed(SEED)

total = NUM_CALIBRATION_SAMPLES
gsm_n = int(total * 0.20)
manta_n = total - gsm_n

print(f"[INFO] calib samples: total={total}, manta={manta_n}, gsm8k={gsm_n}")

print("[INFO] Loading MANTA...")
manta = load_dataset(MANTA_ID, split=MANTA_SPLIT)

print("[INFO] Loading GSM8K...")
gsm8k = load_dataset(GSM8K_ID, GSM8K_CONFIG, split=GSM8K_SPLIT)

# 셔플 후 필요한 개수만 선택
manta = manta.shuffle(seed=SEED).select(range(min(manta_n, len(manta))))
gsm8k = gsm8k.shuffle(seed=SEED).select(range(min(gsm_n, len(gsm8k))))

# 텍스트 컬럼 통일
manta = manta.map(lambda ex: {"text": pick_text(ex)}, remove_columns=manta.column_names)
gsm8k = gsm8k.map(lambda ex: {"text": pick_text(ex)}, remove_columns=gsm8k.column_names)

calib = concatenate_datasets([manta, gsm8k]).shuffle(seed=SEED)

print("[INFO] calib dataset ready:", calib)
print("[INFO] example:\n", calib[0]["text"][:300])

[INFO] calib samples: total=1024, manta=820, gsm8k=204
[INFO] Loading MANTA...
[INFO] Loading GSM8K...


Map: 100%|██████████| 204/204 [00:00<00:00, 30199.34 examples/s]

[INFO] calib dataset ready: Dataset({
    features: ['text'],
    num_rows: 1024
})
[INFO] example:
 {'id': 'f2c62c5d2bb36fc1cc01258eac61b44d', 'conversations': [{'content': 'For the lesson on the Biography and Political Career of Judy Chan Kapui within the unit on Legislative Council Members, it is necessary to assume that _____ in order to argue that her legislative achievements significantly inf


# Load

In [9]:
# =========================================
# [블록 4] Load tokenizer/model
# =========================================
print("[INFO] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print("[INFO] Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,     # 제출 서버(vLLM)도 float16 기반이므로 여기서 맞추는 게 안전
    trust_remote_code=True,
)

model.eval()
print("[INFO] Model/tokenizer loaded")

[INFO] Loading tokenizer...
[INFO] Loading model...
[INFO] Model/tokenizer loaded


# Recipe

In [ ]:
# =========================================
# [블록 5] Build recipe (mixed precision)
# - o_proj, down_proj : W8A16
# - others Linear     : W4A16
# - last 2 layers protected (fp16)
# =========================================
# 마지막 레이어 인덱스 계산(모델 레이어 수가 다를 수 있으니 실제 len으로 계산)
num_layers = len(model.model.layers)
last_ids = list(range(num_layers - PROTECT_LAST_N_LAYERS, num_layers))
last_re = "|".join(str(i) for i in last_ids)
IGNORE_LAST_LAYERS_REGEX = f"re:^model\\.layers\\.({last_re})\\."

print(f"[INFO] detected num_layers={num_layers}, protect(last {PROTECT_LAST_N_LAYERS})={last_ids}")
print(f"[INFO] ignore regex = {IGNORE_LAST_LAYERS_REGEX}")

# (1) 고비트(W8A16) 타겟: self_attn.o_proj + mlp.down_proj
# regex 타겟을 쓰면 레이어별로 자동 매칭됨
HI_TARGETS = [
    r"re:.*self_attn\.o_proj$",
    r"re:.*mlp\.down_proj$",
]

# (2) 저비트(W4A16) 타겟: 모든 Linear
MAIN_TARGETS = ["Linear"]

# ignore 리스트: 항상 보호 + 마지막 2개 레이어 전부 보호 + (W8으로 갈 애들은 W4에서 제외)
IGNORE_FOR_MAIN = [
    *PROTECT_ALWAYS,
    IGNORE_LAST_LAYERS_REGEX,
    r"re:.*self_attn\.o_proj$",
    r"re:.*mlp\.down_proj$",
]

# ignore 리스트: W8 대상에서도 마지막 2개 레이어는 보호
IGNORE_FOR_HI = [
    *PROTECT_ALWAYS,          # 원하면 여기서 embed/lm_head 빼도 되지만 보통 유지 권장
    IGNORE_LAST_LAYERS_REGEX,
]

recipe = [
    # 먼저 W8A16 (민감/에러 집중 모듈)
    GPTQModifier(
        scheme=SCHEME_HI,
        targets=HI_TARGETS,
        ignore=IGNORE_FOR_HI,
    ),
    # 그 다음 W4A16 (나머지 Linear)
    GPTQModifier(
        scheme=SCHEME_MAIN,
        targets=MAIN_TARGETS,
        ignore=IGNORE_FOR_MAIN,
    ),
]

print("[INFO] recipe ready")

[INFO] detected num_layers=30, protect(last 2)=[28, 29]
[INFO] ignore regex = re:^model\.layers\.(28|29)\.
[INFO] recipe ready


# Run / Save

In [ ]:
# =========================================
# [블록 6] Run oneshot GPTQ and save
# =========================================
os.makedirs(OUT_DIR, exist_ok=True)

print("[INFO] Running oneshot GPTQ... (this may take a while)")
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=calib,                      # 80/20 혼합 데이터
    num_calibration_samples=total,      # 이미 calib를 total로 만들었지만, 명시적으로 동일 값
    max_seq_length=MAX_SEQUENCE_LENGTH,
    output_dir=OUT_DIR,
    recipe=recipe,
)

print(f"[DONE] Saved to: {OUT_DIR}")

[INFO] Running oneshot GPTQ... (this may take a while)


Tokenizing: 100%|██████████| 1024/1024 [00:01<00:00, 898.18 examples/s]

2026-02-21T01:45:43.825695+0900 | reset | INFO - Compression lifecycle reset
2026-02-21T01:45:43.826926+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-21T01:45:43.846879+0900 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-02-21T01:45:43.847156+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



(1/31): Calibrating: 100%|██████████| 1024/1024 [00:04<00:00, 248.61it/s]

2026-02-21T01:45:48.620343+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.o_proj using 1024 samples


...

2026-02-21T01:51:38.349672+0900 | compress | METRIC - time 0.36s
2026-02-21T01:51:38.350229+0900 | compress | METRIC - error 3344.08
2026-02-21T01:51:38.350803+0900 | compress | METRIC - GPU 0 | usage: 6.95% | total memory: 51 GB
2026-02-21T01:51:38.351020+0900 | compress | METRIC - GPU 1 | usage: 4.59% | total memory: 51 GB
2026-02-21T01:51:38.351236+0900 | compress | METRIC - GPU 2 | usage: 4.59% | total memory: 51 GB
2026-02-21T01:51:38.351413+0900 | compress | METRIC - GPU 3 | usage: 4.18% | total memory: 51 GB
2026-02-21T01:51:38.351611+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-21T01:51:38.351991+0900 | compress_modules | INFO - Quantizing model.layers.27.self_attn.k_proj using 1024 samples
2026-02-21T01:51:38.708568+0900 | compress | METRIC - time 0.36s
2026-02-21T01:51:38.708998+0900 | compress | METRIC - error 870.63
2026-02-21T01:51:38.709469+0900 | compress | METRIC - GPU 0 | usage: 6.95% | total memory: 51 GB
2026-02-21T01:51:38.709685+0900 | com

(31/31): Propagating: 100%|██████████| 1024/1024 [00:00<00:00, 3259.98it/s]

2026-02-21T01:51:47.487027+0900 | finalize | INFO - Compression lifecycle finalized for 2 modifiers
2026-02-21T01:51:47.502773+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.



Compressing model: 196it [00:00, 417.04it/s]


[DONE] Saved to: Z:/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2


# Vllm Load TEST

In [13]:
from vllm import LLM

MODEL_PATH = "/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2"

llm = LLM(
    model=MODEL_PATH,
    trust_remote_code=True,
    gpu_memory_utilization=0.85,
    max_model_len=2048,
)
print("vLLM load OK")

INFO 02-21 02:16:07 [utils.py:261] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-21 02:16:07 [model.py:541] Resolved architecture: Exaone4ForCausalLM
INFO 02-21 02:16:07 [model.py:1561] Using max model len 2048


2026-02-21 02:16:07,788	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 02-21 02:16:07 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-21 02:16:07 [vllm.py:624] Asynchronous scheduling is enabled.


...

(EngineCore_DP0 pid=456481) INFO 02-21 02:16:13 [default_loader.py:291] Loading weights took 0.16 seconds
(EngineCore_DP0 pid=456481) INFO 02-21 02:16:13 [gpu_model_runner.py:4130] Model loading took 1.2 GiB memory and 0.544587 seconds
(EngineCore_DP0 pid=456481) INFO 02-21 02:16:17 [backends.py:812] Using cache directory: /home/kimgw200/.cache/vllm/torch_compile_cache/ed826f84a3/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=456481) INFO 02-21 02:16:17 [backends.py:872] Dynamo bytecode transform time: 3.73 s
(EngineCore_DP0 pid=456481) INFO 02-21 02:16:20 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=456481) INFO 02-21 02:16:24 [backends.py:319] Compiling a graph for compile range (1, 8192) takes 4.17 s
(EngineCore_DP0 pid=456481) INFO 02-21 02:16:24 [monitor.py:34] torch.compile takes 7.90 s in total
(EngineCore_DP0 pid=456481) INFO 02-21 02:16:25 [gpu_worker.py:356] Available KV cache memory: 38.05 GiB
(EngineCore_DP0 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:00<00:00, 61.07it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 66.01it/s]


(EngineCore_DP0 pid=456481) INFO 02-21 02:16:27 [gpu_model_runner.py:5063] Graph capturing finished in 2 secs, took 0.54 GiB
(EngineCore_DP0 pid=456481) INFO 02-21 02:16:27 [core.py:272] init engine (profile, create kv cache, warmup model) took 13.36 seconds


(EngineCore_DP0 pid=456481) The tokenizer you are loading from '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore_DP0 pid=456481) INFO 02-21 02:16:27 [vllm.py:624] Asynchronous scheduling is enabled.
INFO 02-21 02:16:27 [llm.py:343] Supported tasks: ['generate']
vLLM load OK


# Gsm8k TEST

In [15]:
import os
import subprocess

MODEL_PATH = "/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2"

# tokenizers fork 경고 끄기(점수엔 영향 X)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

cmd = [
    "python", "-m", "lm_eval",
    "--model", "vllm",
    "--model_args", f"pretrained={MODEL_PATH},gpu_memory_utilization=0.85,max_model_len=2048",
    "--tasks", "gsm8k",
    "--num_fewshot", "5",
    "--apply_chat_template",
    "--batch_size", "auto",
]

print("RUN:", " ".join(cmd))
subprocess.run(cmd, check=False)

RUN: python -m lm_eval --model vllm --model_args pretrained=/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2,gpu_memory_utilization=0.85,max_model_len=2048 --tasks gsm8k --num_fewshot 5 --apply_chat_template --batch_size auto


2026-02-21:02:21:54 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-21:02:21:57 INFO     [_cli.run:376] Selected Tasks: ['gsm8k']
2026-02-21:02:21:57 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-21:02:21:57 INFO     [evaluator:236] Initializing vllm model, with arguments: {'pretrained': '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2', 'gpu_memory_utilization': 0.85, 'max_model_len': 2048}


INFO 02-21 02:21:59 [utils.py:261] non-default args: {'seed': 1234, 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2'}
INFO 02-21 02:21:59 [model.py:541] Resolved architecture: Exaone4ForCausalLM
INFO 02-21 02:21:59 [model.py:1561] Using max model len 2048
INFO 02-21 02:22:00 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-21 02:22:00 [vllm.py:624] Asynchronous scheduling is enabled.


The tokenizer you are loading from '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore_DP0 pid=458402) INFO 02-21 02:22:05 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2', speculative_config=None, tokenizer='/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=compressed-tensors, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_

...

(EngineCore_DP0 pid=458402) The tokenizer you are loading from '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore_DP0 pid=458402) INFO 02-21 02:22:17 [vllm.py:624] Asynchronous scheduling is enabled.
INFO 02-21 02:22:17 [llm.py:343] Supported tasks: ['generate']


The tokenizer you are loading from '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 625682.76 examples/s]
2026-02-21:02:22:23 INFO     [tasks:700] Selected tasks:
2026-02-21:02:22:23 INFO     [tasks:691] Task: gsm8k (gsm8k/gsm8k.yaml)
2026-02-21:02:22:23 INFO     [evaluator:314] gsm8k: Using g

vllm ({'pretrained': '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2', 'gpu_memory_utilization': 0.85, 'max_model_len': 2048}), gen_kwargs: ({}), limit: None, num_fewshot: 5, batch_size: auto
|Tasks|Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|-----|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k|      3|flexible-extract|     5|exact_match|↑  |0.2274|±  |0.0115|
|     |       |strict-match    |     5|exact_match|↑  |0.0159|±  |0.0034|



CompletedProcess(args=['python', '-m', 'lm_eval', '--model', 'vllm', '--model_args', 'pretrained=/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2,gpu_memory_utilization=0.85,max_model_len=2048', '--tasks', 'gsm8k', '--num_fewshot', '5', '--apply_chat_template', '--batch_size', 'auto'], returncode=0)

In [17]:
clear gpu()

# Speed Test

In [19]:
!vllm bench throughput \
  --model /home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2 \
  --dataset-name random \
  --num-prompts 100 \
  --random-input-len 100 \
  --random-output-len 200 \
  --random-range-ratio 0.0 \
  --seed 42 \
  --gpu-memory-utilization 0.85 \
  --max-model-len 2048

When dataset path is not set, it will default to random dataset
/home/kimgw200/ISSP_kms/LG_LLM_COMP/venv_comp/lib/python3.12/site-packages/vllm/benchmarks/throughput.py:836: UserWarning: Both --prefix-len and --random-prefix-len are specified. The random version (--random-prefix-len) will be preferred in this run.
  validate_args(args)
The tokenizer you are loading from '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
INFO 02-21 02:44:36 [datasets.py:612] Sampling input_len from [100, 100] and output_len from [200, 200]
INFO 02-21 02:44:36 [utils.py:261] non-default args: {'tokenizer': '/home/kimgw200/ISSP_kms/LG_LLM_COMP/outputs2', 'seed': 42, 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'enable_lora'